In [ ]:
# Main Code

import os
import time
import tempfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp
from numbers import Number
from pysr import PySRRegressor
try:                                  # PySR >=0s.13
    from pysr.sr import SearchTimedOut
except Exception:                     # fall‑back – we only need the name
    class SearchTimedOut(Exception):
        pass

from sympy import simplify
from pyomo.environ import (
    ConcreteModel, Var, Objective, ConstraintList,
    minimize, SolverFactory, Reals,
    sin, cos, exp, log, sqrt, value
)

import os, time
from pyomo.opt import TerminationCondition
from pyomo.common.tempfiles import TempfileManager

SEEDS = [27,28,29,30,31,32]            # will still be bumped each outer iteration
#PYSR_TIMEOUT = 30  # seconds per GP search (5 min)


# --------------------------------------------------------------------------- #
#  Prepare couenne.opt alongside this script / notebook
# --------------------------------------------------------------------------- #
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OPTFILE    = SCRIPT_DIR / "couenne.opt"
T_LIMIT    = 600   # 600 s = 10 min per solve



# --------------------------------------------------------------------------- #
#  welded‑beam evaluator (true simulator)
# --------------------------------------------------------------------------- #
P, L = 6000, 14
tau_max, sigma_max = 13600, 30000
def welded_beam(x):
    x1,x2,x3,x4 = x
    f = 1.10471*x1**2*x2 + 0.04811*x3*x4*(L + x2)
    R = np.sqrt(x2**2/4 + (x1+x3)**2/4)
    M = P*(L + x2/2)
    J = 2*np.sqrt(0.5)*x1*x2*(x2**2/12 + 0.25*(x1+x3)**2)
    tau_p = P/(np.sqrt(2)*x1*x2)
    tau_pp = M*R/J
    tau = np.sqrt(tau_p**2 + tau_pp**2 + tau_p*tau_pp*x2/R)
    g1 = tau/tau_max - 1
    sigma = 6*P*L/(x4*x3**2)
    g2 = sigma/sigma_max - 1
    g3 = (x1 - x4)/(5.0 - 0.125)
    Pc = 64746.022*(1 - 0.0282346*x3)*x3*x4**3
    g4 = P/Pc - 1
    return f, np.array([g1,g2,g3,g4])

# --------------------------------------------------------------------------- #
#  SymPy  →  Pyomo translator (unchanged)
# --------------------------------------------------------------------------- #
def sympy_to_pyomo(expr, vmap):
    if getattr(expr, "is_number", False):
        return float(expr)
    #if expr.is_Number:        return float(expr)
    if expr in vmap:          return vmap[expr]
    if expr.is_Symbol:        raise ValueError(f"Unmapped symbol {expr}")

    args = [sympy_to_pyomo(a, vmap) for a in expr.args]
    f = expr.func
    if f is sp.Add:
        out = args[0]
        for a in args[1:]: out = out + a
        return out
    if f is sp.Mul:
        out = args[0]
        for a in args[1:]: out = out * a
        return out
    if f is sp.Pow:           return args[0] ** args[1]
    if f is sp.sin:  return sin(args[0])
    if f is sp.cos:  return cos(args[0])
    if f is sp.exp:  return exp(args[0])
    if f is sp.log:  return log(args[0])
    if f is sp.sqrt: return sqrt(args[0])
    if f is sp.Abs:  return abs(args[0])
    raise NotImplementedError(f"SymPy op {f} not implemented")

# --------------------------------------------------------------------------- #
#  one long‑lived PySR regressor  (compiles Julia only once)
# --------------------------------------------------------------------------- #
m_gp = PySRRegressor(
    maxsize=20,
    niterations=60,
    binary_operators=["+","-","*","/"],
    unary_operators=["sin","cos","exp","log","inv(x)=1/x"],
    extra_sympy_mappings={"inv": lambda z: 1/z},
    elementwise_loss="loss(p,t)=(p-t)^2",
    #deterministic=True,                # remove “non‑deterministic” warnings
    parallelism="multiprocessing",     # spawn Julia workers once
    procs=8,                           # adapt to your machine
    progress=False, verbosity=0,
    #timeout_in_seconds=PYSR_TIMEOUT,   # <-- watchdog
    warm_start=False                   # fresh population every .fit()
)

# --------------------------------------------------------------------------- #
#  iterative optimisation loop
# --------------------------------------------------------------------------- #
n_iter   = 30
sym_vars = sp.symbols("x0 x1 x2 x3")
x_lower  = [0.125, 0.1, 0.1, 0.125]
x_upper  = [5.0,   10.0, 10.0, 5.0]
tol      = 1e-6


# get absolute path of your project root
BASE_DIR = os.getcwd()

# create an absolute “results” folder
result_folder = os.path.join(BASE_DIR, "results")

def fit_pysr(y):
    """Fit GP; if it times out keep whatever equations_ we have."""
    global SEED
    m_gp.set_params(random_state=SEED)   # vary seed run‑to‑run
    try:
        m_gp.fit(X_all, y)
    except SearchTimedOut:
        print("  [PySR] GP search timed‑out – using best expression so far.")
    #except Exception as e:
        #print("  [PySR] GP crashed:", e, "\n    Re‑spawning Julia workers.")
        #m_gp.clean(); m_gp.spawn_new()
        #raise
    if m_gp.equations_.empty:
        raise RuntimeError("PySR produced no equations.")
    idx = m_gp.equations_.sort_values("loss").index[0]
    return simplify(m_gp.sympy(idx))

os.makedirs(result_folder, exist_ok=True)

for SEED in SEEDS:
    # --------------------------------------------------------------------------- #
    #  load initial data
    # --------------------------------------------------------------------------- #
    dfG = pd.read_csv("welded_beam_constraints.csv")
    dfF = pd.read_csv("welded_beam_objective.csv")
    X_all = dfF[[f"x{i+1}" for i in range(4)]].values.copy()
    F_all = dfF["f"].values.copy()
    G_all = dfG[[f"g{i+1}" for i in range(4)]].values.copy()

    best_expr_f  = None
    best_expr_gs = None
    best_metric  = float("inf")
    feasible_found = False
    seed_folder = os.path.join(result_folder, f"seed_{SEED:02d}")
    os.makedirs(seed_folder, exist_ok=True)

    for it in range(1, n_iter+1):
        print(f"\n--- iteration {it} ---")

        # 1) fit PySR surrogates
        expr_f  = fit_pysr(F_all)
        expr_gs = [fit_pysr(G_all[:, i]) for i in range(4)]

        print("expression for f:", expr_f)
        print("expressions for g1…g4:", expr_gs)

        # 2) build & solve surrogate MINLP with Couenne
        model = ConcreteModel()
        model.x = Var(range(4), domain=Reals)
        for i, (lb, ub) in enumerate(zip(x_lower, x_upper)):
            model.x[i].setlb(lb); model.x[i].setub(ub)
        vmap = {sym_vars[i]: model.x[i] for i in range(4)}

        model.obj  = Objective(expr=sympy_to_pyomo(expr_f, vmap), sense=minimize)
        model.cons = ConstraintList()
        for k, gi in enumerate(expr_gs, start=1):
            lhs = sympy_to_pyomo(gi, vmap)
            if isinstance(lhs, Number):
                if lhs <= 0:  continue
                raise RuntimeError(f"g{k} infeasible surrogate ({lhs:+.4g})")
            model.cons.add(lhs <= 0)



        #solver = SolverFactory("couenne",
                            #executable="D:\\ANKUSH\\Couenne\\couenne.exe")
        #solver.options["bonmin.time_limit"] = 600 
        #solver.options["max_cpu_time"] = 600

        #solver.solve(model, tee=False,)

        # 3) SCIP sollution

        solver = SolverFactory("scip")
        solver.options['limits/time'] = 600  # Time limit in seconds
        solver.options['presolving/maxrounds'] = 5  # Presolving rounds

        t0 = time.perf_counter()
        results = solver.solve(
            model,
            tee=False,
            keepfiles=True,
            load_solutions=True
        )
        elapsed = time.perf_counter() - t0

        print(f"   → SCIP done in {elapsed:.1f}s, "
              f"term={results.solver.termination_condition}")


        # 3) evaluate true function at surrogate optimum
        x_star   = np.array([value(model.x[i]) for i in range(4)])
        f_true, g_true = welded_beam(x_star)
        feas = (g_true <= tol).all()

        print(" x* =", np.round(x_star, 4),
            "| f_true =", round(f_true, 4),
            "| g_true =", np.round(g_true, 4),
            "| feasible?", feas)

        # 4) update best record per your rule
        if not feasible_found:
            violation = np.sum(np.maximum(g_true, 0.0))
            if violation < best_metric:
                best_metric  = violation
                best_expr_f  = expr_f
                best_expr_gs = expr_gs
            if feas:
                feasible_found = True
                best_metric    = f_true
                best_expr_f    = expr_f
                best_expr_gs   = expr_gs
        else:
            if feas and f_true < best_metric:
                best_metric  = f_true
                best_expr_f  = expr_f
                best_expr_gs = expr_gs

        # 5) append to data & bump seed
        X_all = np.vstack([X_all, x_star])
        F_all = np.append(F_all, f_true)
        G_all = np.vstack([G_all, g_true])

    # --------------------------------------------------------------------------- #
    #  save all evaluations
    # --------------------------------------------------------------------------- #
    pd.DataFrame(
        np.column_stack([X_all, G_all]),
        columns=[f"x{i+1}" for i in range(4)] + [f"g{i+1}" for i in range(4)]
    ).to_csv(os.path.join(seed_folder, "all_evals_constraints.csv"), index=False)

    pd.DataFrame(
        np.column_stack([X_all, F_all]),
        columns=[f"x{i+1}" for i in range(4)] + ["f"]
    ).to_csv(os.path.join(seed_folder, "all_evals_objective.csv"), index=False)

    print("\nSaved all_evals_constraints.csv  and  all_evals_objective.csv")

    print("\n=== Best surrogate found ===")
    print("Feasible ever?", feasible_found)
    if not feasible_found:
        print("  → best total violation =", best_metric)
    else:
        print("  → best f_true =", best_metric)
    print("  objective surrogate:", best_expr_f)
    for i, g in enumerate(best_expr_gs, 1):
        print(f"  constraint {i} surrogate:", g)

    # --------------------------------------------------------------------------- #
    #  pickle best surrogate expressions
    # --------------------------------------------------------------------------- #
    import pickle
    with open(os.path.join(seed_folder, "best_surrogate.pkl"), "wb") as f:
        pickle.dump({
            "expr_f":         best_expr_f,
            "expr_gs":        best_expr_gs,
            "feasible_found": feasible_found,
            "best_metric":    best_metric
        }, f)
    print("Saved best surrogate expressions to best_surrogate.pkl")
